# Self-Information (Surprisal) -- Interactive Notebook

**Concept 35 of the math-foundations map.**

We define and explore
$$ I(x) = -\log p(x), $$
the *surprisal* of an outcome under a probability distribution.
Rare outcomes carry more information; the log base sets the units
(bits, nats, or hartleys).

Prereq: `23-distributions`. Forward link: `36-shannon-entropy`.


In [ ]:
import math

def I(p, base=2):
    """Self-information -log_base(p). Default base 2 -> bits."""
    if not 0.0 < p <= 1.0:
        raise ValueError(f'p must be in (0, 1], got {p}')
    return -math.log(p) / math.log(base)

# Sanity check: certainty carries zero information.
print(f'I(p=1)   = {I(1.0):.4f} bits')
print(f'I(p=1/2) = {I(0.5):.4f} bits  # one fair coin flip')
print(f'I(p=1/8) = {I(0.125):.4f} bits')


## 1. Surprisal is monotonically decreasing in $p$

We tabulate $I(p)$ across a fine grid of probabilities. As $p \to 0$, surprisal diverges; as $p \to 1$, it falls to zero.


In [ ]:
ps = [10**(-k) for k in range(0, 8)]
print(f"{'p':>10} | {'I(p) bits':>12} | {'I(p) nats':>12}")
print('-' * 42)
for p in ps:
    print(f'{p:>10.1e} | {I(p, 2):>12.4f} | {I(p, math.e):>12.4f}')


## 2. Additivity for independent events

**Theorem.** If $A, B$ are independent, $I(A \cap B) = I(A) + I(B)$.

**Proof.** $\Pr(A \cap B) = p_A p_B$, so $-\log(p_A p_B) = -\log p_A - \log p_B$. $\square$

Below we verify this for several pairs of independent biased-coin flips.


In [ ]:
pairs = [(0.5, 0.5), (0.9, 0.3), (0.1, 0.8), (0.25, 0.75)]
print(f"{'p_A':>6} {'p_B':>6} | {'I(AB) direct':>14} | {'I(A)+I(B)':>12} | match")
print('-' * 56)
for pA, pB in pairs:
    lhs = I(pA * pB)
    rhs = I(pA) + I(pB)
    print(f'{pA:>6.2f} {pB:>6.2f} | {lhs:>14.6f} | {rhs:>12.6f}'
          f' | {math.isclose(lhs, rhs)}')


## 3. Expectation of surprisal = Shannon entropy

Self-information is a *random variable* once we view $X \sim p$.
Its expectation is the **Shannon entropy** (concept 36):
$$ H(p) = \mathbb{E}_{x \sim p}[I(x)] = -\sum_x p(x) \log p(x). $$
We verify this numerically across a sweep of Bernoulli parameters and plot the entropy curve.


In [ ]:
def H(probs, base=2):
    return sum(p * I(p, base) for p in probs if p > 0)

def E_I(probs, base=2):
    return sum(p * I(p, base) for p in probs if p > 0)

print(f"{'p(H)':>6} | {'E[I(X)]':>10} | {'H(p)':>10} | match?")
print('-' * 42)
for pH in [0.01, 0.1, 0.25, 0.5, 0.75, 0.9, 0.99]:
    probs = [pH, 1.0 - pH]
    print(f'{pH:>6.2f} | {E_I(probs):>10.6f} | {H(probs):>10.6f}'
          f' | {math.isclose(E_I(probs), H(probs))}')


## 4. ASCII plot: the binary entropy function

$H_2(p) = -p \log_2 p - (1-p) \log_2 (1-p)$ peaks at $p = 1/2$ with value $1$ bit.
Surprisal averaged over a fair coin is the *most information* a single binary outcome can carry.


In [ ]:
WIDTH = 50
for k in range(0, 21):
    p = k / 20
    if 0 < p < 1:
        h = H([p, 1 - p])
    else:
        h = 0.0
    bar = '#' * int(round(h * WIDTH))
    print(f'p={p:0.2f}  H={h:0.4f}  |{bar}')


## 5. ML connection: per-token cross-entropy *is* self-information

For a language model $q$, the per-token loss
$$ \mathcal{L}_t = -\log q(x_t \mid x_{<t}) $$
is literally the self-information that $q$ assigns to the observed token.
Below we map a list of model probabilities into surprisals and report the
average (cross-entropy) and the perplexity $2^{\bar{\mathcal{L}}}$.


In [ ]:
model_probs = [0.5, 0.2, 0.8, 0.05, 0.9, 0.3, 0.001, 0.6]
losses = [I(q) for q in model_probs]
for q, l in zip(model_probs, losses):
    print(f'  q = {q:0.3f}   loss = {l:7.4f} bits')
avg = sum(losses) / len(losses)
ppl = 2 ** avg
print(f'\nmean cross-entropy = {avg:.4f} bits/token')
print(f'perplexity         = {ppl:.4f}')


## Summary

- $I(x) = -\log p(x)$ is the unique (up to base) measure of surprisal satisfying continuity, monotonicity, and additivity for independent events.
- We verified additivity numerically: $I(AB) = I(A) + I(B)$ for independent $A, B$.
- We verified that the expectation $\mathbb{E}[I(X)]$ matches Shannon entropy $H(p)$.
- Per-token cross-entropy loss in language modeling is precisely a self-information; perplexity exponentiates its mean.
